# Demo: Forecasts loader and quick plots

This notebook loads the forecast CSVs produced by the Checkpoint 1 demo and visualizes actual vs forecast volatility for each ticker. It also computes simple metrics (RMSE/MAE) and saves plots to artifacts/plots.


In [ ]:
# Imports and plotting configuration
import warnings
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set(style="whitegrid")
warnings.filterwarnings("ignore")


In [ ]:
# Reproducibility and display options
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

pd.options.display.max_rows = 20
pd.options.display.max_columns = 40
pd.options.display.width = 120


In [ ]:
# Load forecast CSVs produced by the pipeline
base_path = Path("..") / ".." / ".." / "quant_pipeline" / "artifacts" / "forecasts"
# resolve in case notebook is opened from different cwd
base_path = base_path.resolve()
files = sorted(list(base_path.glob("*_forecast.csv")))
print(f"Found {len(files)} forecast files in: {base_path}")

for f in files:
    print('-', f.name)

if not files:
    print("No forecast CSVs found. Run `python -m ml.pipelines.train_pipeline` first.")


In [ ]:
# Inspect and plot each forecast
plot_out = (base_path.parent / "plots").resolve()
plot_out.mkdir(parents=True, exist_ok=True)

summary_rows = []
for f in files:
    df = pd.read_csv(f)
    # standardize date column
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.set_index("date").sort_index()
    # prefer har_sentiment_forecast if present
    pred_col = "har_sentiment_forecast" if "har_sentiment_forecast" in df.columns else "har_forecast"
    if pred_col not in df.columns or "actual_volatility" not in df.columns:
        print(f"Skipping {f.name}: required columns not found")
        continue

    actual = df["actual_volatility"].dropna()
    pred = df[pred_col].reindex(actual.index).fillna(method="ffill").fillna(method="bfill")

    rmse = mean_squared_error(actual, pred, squared=False)
    mae = mean_absolute_error(actual, pred)
    summary_rows.append({"file": f.name, "rmse": float(rmse), "mae": float(mae)})

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(actual.index, actual.values, label="Actual", linewidth=2)
    ax.plot(pred.index, pred.values, label="Forecast", linewidth=1.5)
    ax.set_title(f.name)
    ax.set_ylabel("Volatility")
    ax.legend()
    out_path = plot_out / f.name.replace("_forecast.csv", "_viz.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)
    display(df.head(3))
    print(f"Saved plot: {out_path}, RMSE: {rmse:.6f}, MAE: {mae:.6f}\n")

# summary
if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print("No plots produced.")
